In [192]:
import numpy as np

In [193]:
UP, RIGHT, DOWN, LEFT = 0, 1, 2, 3
DIRS = {UP: (-1, 0), RIGHT: (0, 1), DOWN: (1, 0), LEFT: (0, -1)}
ARROWS = {UP: "↑", RIGHT: "→", DOWN: "↓", LEFT: "←"}

In [194]:
class GridWorld:

    def __init__(
        self,
        rows: int,
        cols: int,
        step_reward: float,
        terminals: dict[tuple[int, int], float],
        walls: set[tuple[int, int]],
        seed = 0,
        noise = 0.0,
    ):
        self.rows = rows
        self.cols = cols
        self.step_reward = step_reward
        self.terminals = terminals
        self.walls = walls
        self.noise = noise
        self.rng = np.random.default_rng(seed)
        self.s2c = [
            (r, c)
            for r in range(self.rows)
            for c in range(self.cols)
            if (r, c) not in self.walls
        ]
        self.nS = len(self.s2c)
        self.nA = 4
        self.c2s = {cell: i for i, cell in enumerate(self.s2c)}
        self.nonterminal = [
            s for s, cell in enumerate(self.s2c) if cell not in self.terminals
        ]

    def seed(self, v: int):
        self.rng = np.random.default_rng(v)

    def reset(self, s=None):
        """start a new episode; returns the start state"""
        if s is None:
            s = int(self.rng.choice(self.nonterminal))
        return s

    def move(self, s: int, a: int):
        cell = self.s2c[s]
        dr, dc = DIRS[a]
        nxt = (cell[0] + dr, cell[1] + dc)
        if (
            not 0 <= nxt[0] < self.rows
            or not 0 <= nxt[1] < self.cols
            or nxt in self.walls
        ):
            nxt = cell
        reward = self.terminals.get(nxt, self.step_reward)
        return self.c2s[nxt], reward             

    def transitions(self, s: int, a: int):
        result = []
        outcomes = {
            a: 1 - self.noise,
            (a + 1) % 4: self.noise / 2,
            (a - 1) % 4: self.noise / 2,
        }
        for act, prob in outcomes.items():
            ns, r = self.move(s, act)
            result.append((prob, ns, r))
        return result

    def sample_step(self, s: int, a: int):
        result = self.transitions(s, a)
        i = self.rng.choice(len(result), p=[p for p, _, _ in result])
        _, ns, r = result[i]
        done = self.s2c[ns] in self.terminals
        return ns, r, done

    def cell_repr(self, r, c):
        cell = (r, c)
        if cell in self.terminals:
            return f'{self.terminals[cell]:+}'
        elif cell in self.walls:
            return '#'
        else:
            return '·'

    def render(self):
        for r in range(self.rows):
            for c in range(self.cols):
                if r == 0 and c == 0:
                    print(" r/c", end="")
                    print(''.join([f'{v:>3} ' for v in range(self.cols)]))
                if c == 0:
                    print(f'{r:>3} ', end="")
                print(f'{self.cell_repr(r, c):>3} ', end="")
            print()

    def __repr__(self):
        return f'''
Grid(
    rows={self.rows},
    cols={self.cols},
    step_reward={self.step_reward},
    terminals={self.terminals},
    walls={self.walls},
    noise={self.noise},
)
'''

In [195]:
env = GridWorld(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1, (1, 3): -1},
    walls={(1, 1)},
    noise=0.2,
)
env


Grid(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1, (1, 3): -1},
    walls={(1, 1)},
    noise=0.2,
)

In [196]:
env.render()

 r/c  0   1   2   3 
  0   ·   ·   ·  +1 
  1   ·   #   ·  -1 
  2   ·   ·   ·   · 


### value iteration

In [197]:
def read_policy(
    env: GridWorld,
    V: np.ndarray,
    gamma=0.9,
):
    policy = np.zeros((env.nS, env.nA))
    for s in range(env.nS):
        if env.s2c[s] in env.terminals:
            continue
        best_a = int(np.argmax(q_from_v(env, V, s, gamma)))
        policy[s, best_a] = 1.0
    return policy

def render_policy(env: GridWorld, policy: np.ndarray):
    for r in range(env.rows):
        for c in range(env.cols):
            if r == 0 and c == 0:
                print(" r/c", end="")
                print(''.join([f'{v:>3} ' for v in range(env.cols)]))
            if c == 0:
                print(f'{r:>3} ', end="")  

            cell = (r, c)
            if cell in env.c2s:
                if cell in env.terminals:
                    value = f'{env.terminals[cell]:+}'
                else:
                    value = ARROWS[int(np.argmax(policy[env.c2s[(r, c)]]))]
            else:
                value = '#'
            print(f'{value:>3} ', end='')
        print()    

def show_V(env: GridWorld, V: np.ndarray):
    for r in range(env.rows):
        for c in range(env.cols):
            if r == 0 and c == 0:
                print("  r/c", end="")
                print(''.join([f'{v:>4} ' for v in range(env.cols)]))
            if c == 0:
                print(f'{r:>4} ', end="")            
            cell = (r, c)
            if cell in env.c2s:
                value = round(V[env.c2s[(r, c)]], 2)
            else:
                value = '#'
            print(f'{value:>4} ', end="")
        print()

def q_from_v(
    env: GridWorld,
    V: np.ndarray,
    s: int,
    gamma: float,
):
    q = np.zeros(env.nA)
    if env.s2c[s] in env.terminals:
        return q
    for a in range(env.nA):
        for prob, ns, r in env.transitions(s, a):
            q[a] += prob * (r + gamma * V[ns])
    return q

def value_iteration(
    env: GridWorld,
    gamma=0.9,
    theta=1e-6,
    max_iters=1000,
    verbose=0,
):
    V = np.zeros(env.nS)
    policy = np.zeros((env.nS, env.nA))
    if verbose >= 2:
        print('---------- value_iteration ------------')
        print('V init')
        show_V(env, V)
        print('-' * 25)    
    delta = float('inf')
    i = 0
    while delta >= theta and i < max_iters:
        delta = 0.0
        V_old = V.copy()
        for s in range(env.nS):
            if env.s2c[s] in env.terminals:
                continue
            q = q_from_v(env, V_old, s, gamma)
            V[s] = np.max(q)
            best_a = int(np.argmax(q))
            policy[s] = np.zeros(env.nA)
            policy[s, best_a] = 1.0
            delta = max(delta, abs(V[s] - V_old[s]))
        if verbose >= 1:
            print(f"iter {i}: delta={delta:.6f}")
        if verbose >= 2:
            show_V(env, V)
            print('-' * 25)
        i += 1
    if verbose >= 2:
        render_policy(env, policy)
        print('-' * 25)
    converged = delta < theta
    if verbose >= 1:
        if converged:
                print(f'value_iteration converged in {i - 1} iterations')
        else:
            print(f"value_iteration did not converge in {max_iters} iterations")
    return policy, V, converged

In [198]:
policy, V, converged = value_iteration(env, verbose=2)

---------- value_iteration ------------
V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=0.800000
  r/c   0    1    2    3 
   0  0.0  0.0  0.8  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 1: delta=0.576000
  r/c   0    1    2    3 
   0  0.0 0.58 0.87  0.0 
   1  0.0    # 0.48  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 2: delta=0.414720
  r/c   0    1    2    3 
   0 0.41 0.73 0.92  0.0 
   1  0.0    # 0.57  0.0 
   2  0.0  0.0 0.34  0.0 
-------------------------
iter 3: delta=0.298598
  r/c   0    1    2    3 
   0 0.56  0.8 0.93  0.0 
   1  0.3    # 0.61  0.0 
   2  0.0 0.25 0.41 0.15 
-------------------------
iter 4: delta=0.237199
  r/c   0    1    2    3 
   0 0.65 0.82 0.94  0.0 
   1 0.46    # 0.63  0.0 
   2 0.24 0.34 0.48 0.21 
-------------------------
iter 5: delta=0.145858
  r/c   0    1    2    3 
   0 0.69

In [199]:
policy = read_policy(env, V)
render_policy(env, policy)

 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ←   ↑   ← 


### sarsa

In [200]:
def epsilon_greedy(
    env: GridWorld, Q: np.ndarray, s: int, eps: float
) -> int:
    if env.rng.random() < eps:
        return int(env.rng.integers(env.nA))
    q = Q[s]
    return int(env.rng.choice(np.flatnonzero(q == q.max())))

In [201]:
def sarsa(
    env: GridWorld,
    num_episodes=20000,
    alpha=0.05,
    gamma=0.9,
    eps0: float = 1.0,
    eps_min: float = 0.05,
) -> np.ndarray:
    Q = np.zeros((env.nS, env.nA))
    for ep in range(num_episodes):
        eps = max(eps_min, eps0 * (1 - ep / num_episodes))   # linear ε anneal
        s = env.reset()
        a = epsilon_greedy(env, Q, s, eps)
        for _ in range(1000):
            ns, r, done = env.sample_step(s, a)
            na = epsilon_greedy(env, Q, ns, eps)     
            target = r + gamma * Q[ns, na] * (1.0 - done)   
            Q[s, a] += alpha * (target - Q[s, a])
            s, a = ns, na                                  
            if done:
                break
    return Q

In [202]:
def policy_from_Q(Q: np.ndarray) -> np.ndarray:
    """Deterministic greedy policy as a one-hot (nS, nA) matrix."""
    pi = np.zeros_like(Q)
    pi[np.arange(len(Q)), Q.argmax(axis=1)] = 1.0
    return pi

In [203]:
Q = sarsa(env)

In [204]:
policy = policy_from_Q(Q)
render_policy(env, policy)

 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ←   ↑   ← 


In [205]:
def V_from_Q(Q: np.ndarray):
    return Q.max(axis=1)

In [206]:
V_sarsa = V_from_Q(Q)
show_V(env, V_sarsa)

  r/c   0    1    2    3 
   0 0.71 0.81 0.95  0.0 
   1 0.61    # 0.61  0.0 
   2 0.52 0.46 0.47 0.29 


In [207]:
show_V(env, V)

  r/c   0    1    2    3 
   0 0.72 0.83 0.94  0.0 
   1 0.63    # 0.64  0.0 
   2 0.55 0.48 0.53 0.31 


### q_learning

In [208]:
def q_learning(
    env: GridWorld,
    num_episodes=20000,
    alpha=0.05,
    gamma=0.9,
    eps0: float = 1.0,
    eps_min: float = 0.05,
) -> np.ndarray:
    Q = np.zeros((env.nS, env.nA))
    for ep in range(num_episodes):
        eps = max(eps_min, eps0 * (1 - ep / num_episodes))
        s = env.reset()
        for _ in range(1000):
            a = epsilon_greedy(env, Q, s, eps)  
            ns, r, done = env.sample_step(s, a)
            target = r + gamma * Q[ns].max() * (1.0 - done) 
            Q[s, a] += alpha * (target - Q[s, a])
            s = ns
            if done:
                break
    return Q

In [209]:
Q = q_learning(env)

In [210]:
policy = policy_from_Q(Q)
render_policy(env, policy)

 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ←  -1 
  2   ↑   ←   ←   ↓ 


In [211]:
V_q_learning = V_from_Q(Q)
show_V(env, V_q_learning)

  r/c   0    1    2    3 
   0  0.7 0.79 0.87  0.0 
   1 0.61    # 0.45  0.0 
   2 0.53 0.48 0.42 0.25 


In [212]:
show_V(env, V)

  r/c   0    1    2    3 
   0 0.72 0.83 0.94  0.0 
   1 0.63    # 0.64  0.0 
   2 0.55 0.48 0.53 0.31 


In [213]:
Q = q_learning(env, eps_min=0.4)
policy = policy_from_Q(Q)
render_policy(env, policy)

 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ←   ↑   ← 
